# FRAP diffusion analysis under confinement (6FP vs 11FP)

Fits single-exponential recovery curves to FRAP measurements (Zeiss 880 Airyscan, intensities exported from Fiji),
derives the recovery half-time and an apparent diffusion coefficient for each cell, and compares the 6FP and 11FP conditions.

1. **Normalization** – double normalization of the bleach region to the whole cell (background-subtracted, pre-bleach = 1).
2. **Fit** – $I(t) = I_0 + I_1\,(1 - e^{-(t - t_{bleach})/\tau})$ on the post-bleach frames.
3. **Derived values** – $t_{1/2} = \tau \ln 2$, mobile fraction $= I_1/(I_0 + I_1)$, and $D = 0.22\,r^2 / t_{1/2}$ with $r$ the bleach-spot radius.
4. **Statistics** – Mann–Whitney U (main test, data are non-normal), plus Welch's t-test / ANOVA, Shapiro–Wilk and Levene for reference.

Set the paths and parameters in the **Settings** cell, then run all cells.

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
from scipy.optimize import curve_fit
from scipy.stats import levene, mannwhitneyu, shapiro
from sklearn.metrics import r2_score

## Settings

In [ ]:
# Folder with one .xlsx per cell, exported from Fiji. Columns (after one header row to skip):
# Index, Time (s), Bleach Region, Control Fluorescence, Background, Whole Cell
DATA_FOLDER = "data/"

# Where per-cell fit plots, the results CSV and summary figures are written
OUTPUT_FOLDER = "output/"

NORMALIZATION = "double"   # "double" (to whole cell) or "single"
PREBLEACH_FRAMES = 10      # frames averaged for the pre-bleach reference
BLEACH_INDEX = 10          # index of the first post-bleach frame
BLEACH_RADIUS_UM = 1.63    # bleach-spot radius in µm -> CHANGE TO MATCH YOUR DATA
R2_THRESHOLD = 0.5         # fits with R² below this are excluded from the statistics

CONDITIONS = ["6FP", "11FP"]
PALETTE = {"6FP": "darkorange", "11FP": "lightcoral"}

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
file_list = sorted(glob.glob(os.path.join(DATA_FOLDER, "*.xlsx")))
if not file_list:
    raise FileNotFoundError(f"No .xlsx files found in {os.path.abspath(DATA_FOLDER)} - "
                            "copy your Fiji exports there or change DATA_FOLDER")
print(f"Found {len(file_list)} files in {os.path.abspath(DATA_FOLDER)}")

## Normalize and fit each FRAP curve

In [ ]:
def frap_model_fixed_bleach(t, I0, I1, z, t_bleach_offset=0):
    '''
    FRAP recovery model with mobile fraction and fixed bleach point
    I0: baseline intensity (immobile fraction)
    I1: mobile fraction amplitude
    z: characteristic time
    t_bleach_offset: offset from first data point (fixed at 0)
    '''
    t_bleach = t[0] + t_bleach_offset  # Bleach happens at first post-bleach frame
    return I0 + I1 * (1 - np.exp(-(t - t_bleach) / z))


def get_condition(filename):
    for condition in CONDITIONS:
        if condition in filename:
            return condition
    return "Other"


def normalize(df):
    pre_bleach_bleach_region = df["Bleach Region"].iloc[:PREBLEACH_FRAMES].mean()
    pre_bleach_control = df["Whole Cell"].iloc[:PREBLEACH_FRAMES].mean()
    pre_bleach_background = df["Background"].iloc[:PREBLEACH_FRAMES].mean()

    df["Single Normalized FRAP"] = (
        (df["Bleach Region"] - df["Background"]) /
        (pre_bleach_bleach_region - pre_bleach_background)
    )
    df["Double Normalized FRAP"] = df["Single Normalized FRAP"] * (
        (pre_bleach_control - pre_bleach_background) /
        (df["Whole Cell"] - df["Background"])
    )
    return df


def initial_guess(t_data, intensity_data, t_bleach):
    # Minimum intensity in first 20 frames after bleach
    min_window = min(20, len(intensity_data))
    I0_guess = np.min(intensity_data[:min_window])

    # Plateau from the last 20% of data
    plateau_start = int(0.8 * len(intensity_data))
    if plateau_start >= len(intensity_data):
        plateau_start = len(intensity_data) - 5
    plateau_intensity = np.median(intensity_data[plateau_start:])

    # Mobile fraction (difference between plateau and bleach)
    I1_guess = plateau_intensity - I0_guess

    # Characteristic time: time to reach ~50% recovery
    z_guess = (max(t_data) - min(t_data)) / 3
    if I1_guess > 0:
        idx_50 = np.where(intensity_data >= I0_guess + 0.50 * I1_guess)[0]
        if len(idx_50) > 0:
            z_guess = t_data[idx_50[0]] - t_bleach

    return [I0_guess, max(I1_guess, 0.01), max(z_guess, 1.0)]


def plot_fit(filename, t_data_full, intensity_data_full, t_data, popt, R2, mobile_fraction, t_bleach):
    t_fit = np.linspace(min(t_data), max(t_data), 1000)
    plateau_value = popt[0] + popt[1]

    plt.figure(figsize=(8, 5))
    plt.scatter(t_data_full, intensity_data_full, color='blue', alpha=0.5, s=30, label="Data")
    plt.plot(t_fit, frap_model_fixed_bleach(t_fit, *popt), color='red', linewidth=2.5,
             label=f"Fit (R²={R2:.3f})")
    plt.axhline(y=plateau_value, color='orange', linestyle='--', linewidth=1.5, alpha=0.7,
                label=f"Plateau={plateau_value:.2f}")
    plt.axvline(x=t_bleach, color='gray', linestyle=':', linewidth=1.5, alpha=0.5,
                label=f"Bleach at {t_bleach:.1f}s")
    plt.xlabel("Time (s)", fontsize=12)
    plt.ylabel(f"{NORMALIZATION.capitalize()} Normalized Intensity", fontsize=12)
    plt.title(f"FRAP Fit: {filename}\nMobile Fraction: {mobile_fraction:.2%}", fontsize=11)
    plt.legend(fontsize=9, loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_FOLDER, filename.replace('.xlsx', '.png')), dpi=300)
    plt.close()


intensity_column = f"{NORMALIZATION.capitalize()} Normalized FRAP"
results = []

for file in file_list:
    filename = os.path.basename(file)
    condition = get_condition(filename)

    df = pd.read_excel(file, skiprows=1)
    df.columns = ["Index", "Time (s)", "Bleach Region", "Control Fluorescence",
                  "Background", "Whole Cell"]
    df = normalize(df)

    t_data_full = df["Time (s)"].values
    intensity_data_full = df[intensity_column].values
    t_bleach_actual = t_data_full[BLEACH_INDEX]

    # Use data starting from bleach point
    t_data = t_data_full[BLEACH_INDEX:]
    intensity_data = intensity_data_full[BLEACH_INDEX:]

    if len(t_data) < 5:
        print(f"⚠️ Not enough data to fit for {filename}")
        continue

    bounds = (
        [0, 0, 0.1],                            # lower bounds
        [1.2, 1.5, max(t_data) - min(t_data)]   # upper bounds
    )

    try:
        popt, pcov = curve_fit(
            frap_model_fixed_bleach, t_data, intensity_data,
            p0=initial_guess(t_data, intensity_data, t_bleach_actual),
            bounds=bounds,
            maxfev=50000,
            method='trf'  # Trust Region Reflective - more robust
        )
    except Exception as e:
        print(f"⚠️ Fit failed for {filename}: {e}")
        continue

    R2 = r2_score(intensity_data, frap_model_fixed_bleach(t_data, *popt))
    mobile_fraction = popt[1] / (popt[0] + popt[1]) if (popt[0] + popt[1]) > 0 else 0
    plateau_value = popt[0] + popt[1]

    results.append({
        "Filename": filename,
        "Condition": condition,
        "I0": popt[0],
        "I1": popt[1],
        "z": popt[2],
        "t_bleach": t_bleach_actual,
        "R2": R2,
        "Mobile_Fraction": mobile_fraction,
        "Plateau_Value": plateau_value
    })

    plot_fit(filename, t_data_full, intensity_data_full, t_data, popt, R2, mobile_fraction, t_bleach_actual)
    print(f"✅ {filename}: R²={R2:.4f}, Mobile Fraction={mobile_fraction:.2%}, Plateau={plateau_value:.3f}")

## Half-time and diffusion coefficient

In [ ]:
fit_results_df = pd.DataFrame(results)
fit_results_df['t_half'] = fit_results_df['z'] * np.log(2)
fit_results_df['D'] = (0.22 * BLEACH_RADIUS_UM**2) / fit_results_df['t_half']
fit_results_df['Immobile_Fraction'] = 1 - fit_results_df['Mobile_Fraction']
fit_results_df['Recovery_Pct'] = fit_results_df.apply(
    lambda r: (r['I1'] / (1 - r['I0'])) * 100 if (1 - r['I0']) > 0 else np.nan, axis=1
)

results_csv_path = os.path.join(OUTPUT_FOLDER, "FRAP_fit_results.csv")
fit_results_df.to_csv(results_csv_path, index=False)
print("✅ Results saved to:", results_csv_path)

# Exclude bad fits from everything below
fit_results_df_filtered = fit_results_df[fit_results_df['R2'] > R2_THRESHOLD].copy()
print(f"Total samples: {len(fit_results_df)}")
print(f"After filtering (R²>{R2_THRESHOLD}): {len(fit_results_df_filtered)}")
fit_results_df

## Overview plots

In [ ]:
figure, axis = plt.subplots(2, 2, figsize=(12, 10))

for ax, (column, title) in zip(axis.flat, [("R2", "R square"), ("D", "D"), ("t_half", "T half")]):
    sns.swarmplot(data=fit_results_df_filtered, x="Condition", y=column, color='gray', order=CONDITIONS,
                  marker='o', edgecolor='black', ax=ax)
    sns.boxplot(data=fit_results_df_filtered, x="Condition", y=column, hue='Condition', palette=PALETTE,
                order=CONDITIONS, ax=ax)
    ax.set_title(title)

axis[1, 1].axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, "Fitting_Calculations.svg"), dpi=300)
plt.show()

In [ ]:
f, ax = plt.subplots(figsize=(7, 7))
sns.swarmplot(data=fit_results_df_filtered, x="Condition", y="R2", color='gray', order=CONDITIONS,
              marker='o', edgecolor='black')
sns.violinplot(data=fit_results_df_filtered, x="Condition", y="R2", hue='Condition', palette=PALETTE,
               order=CONDITIONS, width=0.8)
sns.boxplot(data=fit_results_df_filtered, x="Condition", y="R2", hue='Condition', palette=PALETTE,
            order=CONDITIONS, width=0.3)
plt.ylim(0, 1.2)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, "R_Squares_Distributions.svg"), dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 5))
colors = {"6FP": "#EF9F27", "11FP": "#D85A30"}

for condition, group in fit_results_df_filtered.groupby("Condition"):
    ax.scatter(group["Recovery_Pct"], group["D"], color=colors.get(condition, "gray"),
               s=60, alpha=0.9, label=condition, linewidths=0.8)

ax.set_yscale("log")
ax.set_ylim(top=0.1)
ax.set_xlabel("% recovery", fontsize=12)
ax.set_ylabel("diffusion coefficient [µm²/s]", fontsize=12)
ax.legend(frameon=False, fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis='both', labelsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, "D_vs_Recovery_logscale.svg"), dpi=300)
plt.show()

## Summary statistics

In [ ]:
summary = (
    fit_results_df_filtered.groupby('Condition')[['D', 't_half']]
      .agg(['mean', 'std', 'median', 'count'])
)
summary

## Comparison between conditions

D and t½ are not normally distributed (Shapiro–Wilk below), so the Mann–Whitney U test is the main test.

In [ ]:
a, b = '11FP', '6FP'
df = fit_results_df_filtered
day_col = next((c for c in ['Date', 'date', 'Replicate', 'replicate'] if c in df.columns), None)

def report(col, unit, label, fmt='.0f'):
    d = df.dropna(subset=[col])
    xa = d.loc[d['Condition'] == a, col]
    xb = d.loc[d['Condition'] == b, col]

    summary = d.groupby('Condition')[col].agg(
        n='count', median='median',
        q1=lambda x: x.quantile(0.25), q3=lambda x: x.quantile(0.75))
    if day_col:
        summary['n_days'] = d.groupby('Condition')[day_col].nunique()

    U, p = mannwhitneyu(xa, xb, alternative='two-sided')
    r_rb = 2 * U / (len(xa) * len(xb)) - 1

    print(f"\n=== {label} ===")
    print(summary.round(4))
    print(f"Mann-Whitney U = {U:.1f}, p = {p:.4f}, rank-biserial r = {r_rb:.2f}")
    s = summary
    print(f"{label} was {'higher' if r_rb > 0 else 'lower'} for {a} than for {b} "
          f"(median {s.loc[a,'median']:{fmt}} {unit} [IQR {s.loc[a,'q1']:{fmt}}–{s.loc[a,'q3']:{fmt}}] vs "
          f"{s.loc[b,'median']:{fmt}} {unit} [{s.loc[b,'q1']:{fmt}}–{s.loc[b,'q3']:{fmt}}]; "
          f"n = {len(xa)} and {len(xb)} cells; two-sided Mann–Whitney U test, p = {p:.3g}).")

report('t_half', 's', 'Recovery half-time', fmt='.0f')
report('D', 'µm²/s', 'The diffusion coefficient', fmt='.4f')

### Assumption checks and parametric tests (for reference)

In [ ]:
for column in ["D", "t_half"]:
    groups = {c: df.loc[df["Condition"] == c, column].values for c in CONDITIONS}
    print(f"\n=== {column} ===")
    for c, values in groups.items():
        print(f"Shapiro–Wilk {c}: {shapiro(values)}")
    print(f"Levene: {levene(*groups.values())}")

    anova_result = stats.f_oneway(*groups.values())
    print(f"One-way ANOVA: F = {anova_result.statistic:.4f}, p = {anova_result.pvalue:.4e}")

    t_stat, p_value = stats.ttest_ind(*groups.values(), equal_var=False)  # Welch's t-test
    print(f"Welch's t-test ({' vs '.join(CONDITIONS)}): t = {t_stat:.4f}, p = {p_value:.4e}")